###Feature Engineering

In [ ]:
#Time-based features
df['Time_hours'] = df['Time'] / 3600

#Transaction Frequency Features
df = df.sort_values('Time').reset_index(drop=True)
df['Time_diff'] = df['Time'].diff().fillna(0)

#Rolling window / Trnasaction velocity features
window_sizes = [10,50,100]
for window in window_sizes:
  df[f'amount_rolling_mean_{window}'] = df['Amount'].rolling(window=window, min_periods=1).mean()
  df[f'amount_rolling_std_{window}'] = df['Amount'].rolling(window=window, min_periods=1).std()
  df[f'time_rolling_mean_{window}'] = df['Amount'].rolling(window=window, min_periods=1).mean()

In [ ]:
#Amount based risk features
df['Amount_percentile'] = df['Amount'].rank(pct=True)

#Risk-bands
amount_bins = [0,1,10,100,1000,10000,float('inf')]
amount_labels = ['micro','small','medium','large','very_large','extreme']
df['Amount_category'] = pd.cut(df['Amount'],bins=amount_bins,labels=amount_labels)

#Amount anomaly scores(distance from median in log space)
df['Amount_log'] = np.log1p(df['Amount'])
median_log_amount = df['Amount_log'].median()
df['Amount_deviation'] = np.abs(df['Amount_log'] - median_log_amount)

In [ ]:
#Latent feature combinations

#Sum of absolute values of PCA components
pca_columns = [f'V{i}' for i in range(1,29)]
df['PCA_magnitude'] = np.sqrt((df[pca_columns] ** 2).sum(axis=1))

#Dominant PCA component
df['PCA_max_component'] = df[pca_columns].max(axis=1)

In [ ]:
#Update feature names for modelling
feature_names = (pca_columns +
 ['Time','Amount','Time_hours','Time_diff','Amount_percentile','Amount_log','Amount_deviation','PCA_magnitude','PCA_max_component'] +
                 [col for col in df.columns if 'rolling' in col])

In [ ]:
#Assign features and target
X = df[feature_names].copy()
y = df['Class'].copy()

In [ ]:
#Handle any missing values from the feature engineering
X = X.fillna(X.median())

In [ ]:
#Perform train-test-split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
#Scale features(robust to oyrliers- important for fintechs)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Fraud Rate: {y_train.mean():.3%}")
print(f"Test Fraud Rate: {y_test.mean():.3%}")

Training Fraud Rate: 0.173%
Test Fraud Rate: 0.172%
